# Caderno 03 -- Embeddings Semanticos e Busca Vetorial

**Objetivo:** Demonstrar geracao de embeddings, indexacao FAISS/ChromaDB,
busca semantica por cosseno, busca hibrida com BM25 e comparacao de modelos.

**Rubrica 3:** Embeddings e Busca Vetorial -- 5 itens (geracao, busca,
avaliacao, analise de falhas, justificativa).

### Fluxo
1. Carregar trechos das bulas (Fonte 1 e 2) com extracao de secoes
2. Gerar embeddings com SentenceTransformer
3. Criar indice FAISS (Inner Product = cosseno)
4. Implementar busca semantica
5. Implementar busca hibrida (FAISS + BM25)
6. Comparar dois modelos de embeddings
7. Avaliar recall@5 e analisar falhas


In [1]:
import os, sys, logging, json, re, time
from pathlib import Path
from datetime import datetime

diretorio_logs = Path("logs"); diretorio_logs.mkdir(exist_ok=True)
formato = logging.Formatter("%(asctime)s [%(levelname)s] %(message)s", datefmt="%Y-%m-%d %H:%M:%S")
fh = logging.FileHandler(diretorio_logs / "caderno_03.log", encoding="utf-8"); fh.setFormatter(formato)
ch = logging.StreamHandler(sys.stdout); ch.setFormatter(formato)
registro = logging.getLogger("caderno_03"); registro.setLevel(logging.INFO)
registro.addHandler(fh); registro.addHandler(ch)

registro.info("=" * 60)
registro.info("Caderno 03 -- Embeddings e Busca Vetorial")
registro.info("Inicio: %s", datetime.now().isoformat())

# Caminho dos dados pruned do processador de bulas
DATA_DIR = Path(r"../python-processador-bulas/data/pruned")

def extrair_nome_medicamento(nome_arquivo):
    nome_base = Path(nome_arquivo).stem
    nome_base = re.sub(r"^\d+_", "", nome_base)  # remove prefixo Fonte 1
    nome_base = re.sub(r"_(paciente|profissional)$", "", nome_base, flags=re.IGNORECASE)
    return nome_base.replace("_", " ").strip().lower()

def carregar_trechos_bulas(diretorio_raiz, maximo_por_fonte=2500):
    trechos = []
    contador_arquivos = 0

    padrao_secao_fonte1 = re.compile(r"##\s*([^\n]+)\s*\n(.*?)(?=\n##|\Z)", re.DOTALL)
    padrao_bloco_fonte2 = re.compile(
        r"\[P:\s*INTERA[\w]+\s*MEDICAMENTOSA\??\s*\]\s*\nR:\s*(.*?)(?=\n\[P:|\Z)",
        re.DOTALL | re.IGNORECASE
    )
    padrao_divisao = re.compile(r"(?<=[.!?;])\s+(?=[A-Z\(])")
    palavras_relevantes = ["interac", "intera\u00e7", "precauc", "contraind", "advert", "devo saber"]

    for fonte in ["fonte1", "fonte2"]:
        dir_fonte = diretorio_raiz / fonte
        if not dir_fonte.is_dir():
            registro.warning("Diretorio nao encontrado: %s", dir_fonte)
            continue
        arquivos = sorted(dir_fonte.glob("*.txt"))[:maximo_por_fonte]

        for caminho_arquivo in arquivos:
            contador_arquivos += 1
            nome_medicamento = extrair_nome_medicamento(caminho_arquivo.name)
            try:
                conteudo = caminho_arquivo.read_text(encoding="utf-8")
            except UnicodeDecodeError:
                continue

            if fonte == "fonte1":
                for match_secao in padrao_secao_fonte1.finditer(conteudo):
                    titulo_secao = match_secao.group(1).lower()
                    if not any(p in titulo_secao for p in palavras_relevantes):
                        continue
                    texto_secao = match_secao.group(2).strip()
                    for sentenca in padrao_divisao.split(texto_secao):
                        sentenca = sentenca.strip()
                        if 30 <= len(sentenca) <= 1000:
                            trechos.append({"medicamento": nome_medicamento, "texto": sentenca,
                                           "fonte": fonte, "nome_arquivo": caminho_arquivo.name})
            else:
                for match_bloco in padrao_bloco_fonte2.finditer(conteudo):
                    texto_bloco = match_bloco.group(1).strip()
                    for sentenca in padrao_divisao.split(texto_bloco):
                        sentenca = sentenca.strip()
                        if 30 <= len(sentenca) <= 1000:
                            trechos.append({"medicamento": nome_medicamento, "texto": sentenca,
                                           "fonte": fonte, "nome_arquivo": caminho_arquivo.name})

    registro.info("Trechos carregados: %d (F1:%d F2:%d) de %d arquivos",
        len(trechos),
        sum(1 for t in trechos if t["fonte"]=="fonte1"),
        sum(1 for t in trechos if t["fonte"]=="fonte2"),
        contador_arquivos)
    return trechos

trechos = carregar_trechos_bulas(DATA_DIR)
print(f"Total de trechos: {len(trechos)}")
print(f"Exemplo trecho[0]: {trechos[0]}")


2026-06-23 22:51:07 [INFO] ============================================================
2026-06-23 22:51:07 [INFO] Caderno 03 -- Embeddings e Busca Vetorial
2026-06-23 22:51:07 [INFO] Inicio: 2026-06-23T22:51:07.745691
2026-06-23 22:51:10 [INFO] Trechos carregados: 116152 (F1:105338 F2:10814) de 3482 arquivos
Total de trechos: 116152
Exemplo trecho[0]: {'medicamento': 'etoricoxibe', 'texto': 'ADVERTÊNCIAS E PRECAUÇÕES  \nEfeito cardiovascular  \nEstudos clínicos sugerem que a classe de inibidores seletivos da COX -2 pode estar associada a risco aumentado \nde eventos trombóticos (particularmente IM e AVC), em relação ao placebo e a alguns AINEs (naproxeno).', 'fonte': 'fonte1', 'nome_arquivo': '100290226_etoricoxibe_profissional.txt'}


## 3.1 SentenceTransformer: Geracao de Embeddings

`sentence-transformers` gera representacoes vetoriais densas de textos.
O modelo escolhido foi **paraphrase-multilingual-MiniLM-L12-v2**:

| Caracteristica | Valor |
|----------------|-------|
| Dimensoes | 384 |
| Parametros | ~118M |
| Idiomas | 50+ (PT-BR incluso) |
| Treino | SNLI + MultiNLI + Anotes |

**Por que nao BioBERTpt?** BioBERTpt tem ~110M parametros e 768 dimensoes,
porem NAO tem versao sentence-transformers pronta. O modelo multilingue
e suficiente para o dominio farmaceutico.


In [2]:
import numpy as np
from sentence_transformers import SentenceTransformer

# Modelo 1: paraphrase-multilingual-MiniLM-L12-v2
#   - 384 dimensoes, 118M parametros
#   - Suporta 50+ idiomas (portugues incluso)
#   - Treinado com SNLI+MultiNLI -> bom para semantica geral
modelo_principal = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
registro.info("Modelo paraphrase-multilingual-MiniLM-L12-v2 carregado")
print(f"Dimensão embeddings: {modelo_principal.get_embedding_dimension()}")

# Teste rapido: embeddings de 3 frases
frases_teste = [
    "amoxicilina interage com anticoagulante warfarina",
    "paracetamol nao tem interacao clinicamente relevante",
    "sinvastatina e contraindicada com itraconazol",
]
embeddings_teste = modelo_principal.encode(frases_teste, normalize_embeddings=True)

# Similaridade de cosseno entre pares
from numpy.linalg import norm
def cosseno(a, b):
    return float(np.dot(a, b))

print("\nTeste de similaridade semantica:")
print(f"  Frase 1 vs 2 (mesma classe 0): {cosseno(embeddings_teste[0], embeddings_teste[1]):.4f}")
print(f"  Frase 1 vs 3 (classes diferentes): {cosseno(embeddings_teste[0], embeddings_teste[2]):.4f}")
print(f"  Frase 2 vs 3 (classes diferentes): {cosseno(embeddings_teste[1], embeddings_teste[2]):.4f}")

registro.info("Embeddings de teste gerados OK")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

2026-06-23 22:51:30 [INFO] Modelo paraphrase-multilingual-MiniLM-L12-v2 carregado
Dimensão embeddings: 384

Teste de similaridade semantica:
  Frase 1 vs 2 (mesma classe 0): 0.3316
  Frase 1 vs 3 (classes diferentes): 0.4630
  Frase 2 vs 3 (classes diferentes): 0.4753
2026-06-23 22:51:30 [INFO] Embeddings de teste gerados OK


## 3.2 FAISS: Indice de Busca Vetorial

FAISS (Facebook AI Similarity Search) permite buscar os k-vetores
mais proximos de uma consulta em milhoes de vetores em milissegundos.

**Estratégia:** `IndexFlatIP` (Inner Product) com vetores **normalizados**.
Com normalizacao L2=1, o produto interno e equivalente a similaridade
de cosseno -- mais eficiente que calcular cosseno diretamente.

**Limite:** FAISS e puramente vetorial. Nao suporta metadata
(medicamento, fonte). O trecho e recuperado pelo indice numerico.


In [3]:
import faiss
import numpy as np

def criar_indice_faiss(trechos, modelo_embeddings):
    registro.info("Gerando embeddings para %d trechos...", len(trechos))
    tempo_inicio = time.time()

    textos = [t["texto"] for t in trechos]
    matriz = modelo_embeddings.encode(
        textos,
        batch_size=32,
        show_progress_bar=True,
        normalize_embeddings=True,  # normalizacao L2 -> IP = cosseno
    )

    tempo_emb = time.time() - tempo_inicio
    registro.info("Embeddings: %d x %d em %.1fs", matriz.shape[0], matriz.shape[1], tempo_emb)

    # Indice FAISS: Inner Product com vetores normalizados = cosseno
    dimensao = matriz.shape[1]
    indice_faiss = faiss.IndexFlatIP(dimensao)
    indice_faiss.add(matriz.astype(np.float32))

    registro.info("Indice FAISS: %d vetores indexados", indice_faiss.ntotal)
    return indice_faiss, matriz

indice_faiss, matriz_embeddings = criar_indice_faiss(trechos, modelo_principal)
print(f"Indice FAISS criado: {indice_faiss.ntotal} vetores")
print(f"Matriz embeddings: {matriz_embeddings.shape}")


2026-06-23 22:51:30 [INFO] Gerando embeddings para 116152 trechos...


Batches:   0%|          | 0/3630 [00:00<?, ?it/s]

2026-06-23 22:53:26 [INFO] Embeddings: 116152 x 384 em 115.2s
2026-06-23 22:53:26 [INFO] Indice FAISS: 116152 vetores indexados
Indice FAISS criado: 116152 vetores
Matriz embeddings: (116152, 384)


## 3.3 Busca Semantica

1. Gerar embedding da consulta (normalize=True)
2. FAISS.search(embedding, top_k) -> k mais similares
3. Mapear indices numericos de volta para trechos

**Metrica:** similaridade de cosseno (mesmo que IP com vetores normalizados)


In [4]:
import numpy as np

def buscar_semantica(consulta, modelo_embeddings, indice_faiss, trechos, top_k=5):
    embedding_consulta = modelo_embeddings.encode(
        [consulta], normalize_embeddings=True
    ).astype(np.float32)

    distancias, indices = indice_faiss.search(embedding_consulta, top_k)

    resultados = []
    for rank, (dist, idx) in enumerate(zip(distancias[0], indices[0]), 1):
        if idx < len(trechos):
            resultados.append({
                "rank": rank,
                "score": float(dist),
                "medicamento": trechos[idx]["medicamento"],
                "texto": trechos[idx]["texto"][:200],
                "fonte": trechos[idx]["fonte"],
            })
    return resultados

consultas_teste = [
    "amoxicilina interacao com warfarina",
    "atorvastatina ciclosporina miopatia",
    "sinvastatina contraindicada",
]

print("TESTE DE BUSCA SEMANTICA\n")
for consulta in consultas_teste:
    resultados = buscar_semantica(consulta, modelo_principal, indice_faiss, trechos, top_k=3)
    print(f"Consulta: {consulta}")
    for r in resultados:
        print(f"  #{r['rank']} [{r['medicamento']}] (score={r['score']:.4f})")
        print(f"      {r['texto'][:150]}...")
    print()


TESTE DE BUSCA SEMANTICA

Consulta: amoxicilina interacao com warfarina
  #1 [omeprazol] (score=0.7576)
      A amoxicilina também apresenta interações medicamentosas....
  #2 [omeprazol] (score=0.7576)
      A amoxicilina  também apresenta interações medicamentosas....
  #3 [omeprazol] (score=0.7576)
      A amoxicilina 
também  apresenta interações  medicamentosas....

Consulta: atorvastatina ciclosporina miopatia
  #1 [pitavastatina calcica] (score=0.7248)
      INTERAÇÕES MEDICAMENTOSAS  
Ciclosporina:  a ciclosporina aumentou significantement e a exposição à pitavastatina....
  #2 [sinergen] (score=0.7136)
      Pode causar aumento de toxicidade à ciclosporina: ciclosporina;...
  #3 [metronidazol] (score=0.7101)
      Ciclosporina: risco de aumento dos níveis plasmáticos de ciclosporina....

Consulta: sinvastatina contraindicada
  #1 [oxalato de escitalopram] (score=0.7223)
      SUA HABILIDADE E 
ATENÇÃO PODEM ESTAR PREJUDICADAS.  
  
5....
  #2 [oxalato de escitalopram] (score=0

## 3.4 Busca Hibrida: FAISS + BM25

BM25 (Best Matching 25) e um algoritmo classico de IR baseado em
frequencia de termos. E forte em correspondência exata de palavras.

**Abordagem hibrida:** Combinar scores semanticos (FAISS) e
keyword-matching (BM25) com peso configuravel:
`score_final = 0.6 * score_semantico + 0.4 * score_bm25_normalizado`

**Vantagens:**
- FAISS: captura semantica (sinonimos, variacao linguistica)
- BM25: recall em termos especificos (nomes de farmacos)


In [5]:
import numpy as np
from rank_bm25 import BM25Okapi
import re

# Tokenizador simples para BM25 (palavras minusculas, alfa-numericas)
def tokenizar(texto):
    return re.findall(r"\b\w+\b", texto.lower())

def criar_indice_bm25(trechos):
    textos = [t["texto"] for t in trechos]
    tokens = [tokenizar(t) for t in textos]
    bm25 = BM25Okapi(tokens)
    registro.info("Indice BM25 criado: %d documentos", len(tokens))
    return bm25, tokens

def buscar_bm25(consulta, bm25, trechos, top_k=5):
    tokens_consulta = tokenizar(consulta)
    scores = bm25.get_scores(tokens_consulta)
    indices_top = np.argsort(scores)[::-1][:top_k]

    resultados = []
    for rank, idx in enumerate(indices_top, 1):
        resultados.append({
            "rank": rank,
            "score": float(scores[idx]),
            "medicamento": trechos[idx]["medicamento"],
            "texto": trechos[idx]["texto"][:200],
            "fonte": trechos[idx]["fonte"],
        })
    return resultados

def buscar_hibrida(consulta, modelo_embeddings, indice_faiss, bm25, trechos, top_k=5, peso_semantico=0.6):
    embed_consulta = modelo_embeddings.encode([consulta], normalize_embeddings=True).astype(np.float32)
    _, indices_sem = indice_faiss.search(embed_consulta, top_k * 2)

    tokens_consulta = tokenizar(consulta)
    scores_bm = bm25.get_scores(tokens_consulta)

    # Combinar scores: peso_semantico * score_faiss + (1-peso) * score_bm25 normalizado
    score_sem_normalizado = scores_bm.max() if scores_bm.max() > 0 else 1.0

    ranking_combinado = {}
    for idx in indices_sem[0]:
        s_sem = float(1 - (np.where(indices_sem[0] == idx)[0][0]) / (2 * top_k))  # rank-based
        s_bm = scores_bm[idx] / score_sem_normalizado
        ranking_combinado[idx] = peso_semantico * s_sem + (1 - peso_semantico) * s_bm

    indices_ordenados = sorted(ranking_combinado, key=ranking_combinado.get, reverse=True)[:top_k]

    resultados = []
    for rank, idx in enumerate(indices_ordenados, 1):
        resultados.append({
            "rank": rank,
            "score_combinado": round(ranking_combinado[idx], 4),
            "medicamento": trechos[idx]["medicamento"],
            "texto": trechos[idx]["texto"][:200],
            "fonte": trechos[idx]["fonte"],
        })
    return resultados

bm25, tokens_bm25 = criar_indice_bm25(trechos)

consulta = "amoxicilina interacao com warfarina anticoagulante"
print("CONSULTA:", consulta)

print("\n--- Semantic (FAISS) ---")
for r in buscar_semantica(consulta, modelo_principal, indice_faiss, trechos, top_k=3):
    print(f"  #{r['rank']} [{r['medicamento']}] score={r['score']:.4f}")

print("\n--- BM25 ---")
for r in buscar_bm25(consulta, bm25, trechos, top_k=3):
    print(f"  #{r['rank']} [{r['medicamento']}] score={r['score']:.4f}")

print("\n--- Hibrida (60% semantica + 40% BM25) ---")
for r in buscar_hibrida(consulta, modelo_principal, indice_faiss, bm25, trechos, top_k=3):
    print(f"  #{r['rank']} [{r['medicamento']}] score={r['score_combinado']:.4f}")


2026-06-23 22:53:28 [INFO] Indice BM25 criado: 116152 documentos
CONSULTA: amoxicilina interacao com warfarina anticoagulante

--- Semantic (FAISS) ---
  #1 [omeprazol] score=0.7720
  #2 [omeprazol] score=0.7720
  #3 [omeprazol] score=0.7720

--- BM25 ---
  #1 [kolpitrat] score=19.5373
  #2 [retemic] score=12.0875
  #3 [benzoilmetronidazol] score=9.5300

--- Hibrida (60% semantica + 40% BM25) ---
  #1 [omeprazol] score=0.7552
  #2 [omeprazol] score=0.6552
  #3 [omeprazol] score=0.5552


## 3.5 Comparacao de Modelos de Embeddings

Dois modelos testados:

| Modelo | Dimensoes | Parametros | Idiomas |
|--------|-----------|------------|---------|
| paraphrase-multilingual-MiniLM-L12-v2 | 384 | ~118M | 50+ |
| all-MiniLM-L6-v2 | 384 | ~22.7M | Ingles |

O modelo multilingue e superior para bulas em portugues por ter
sido treinado com dados multilingues incluindo PT-BR.


In [6]:
from sentence_transformers import SentenceTransformer

# Modelo 2: all-MiniLM-L6-v2
#   - 384 dimensoes, 22.7M parametros (6x menor que o principal)
#   - Mais rapido na geracao de embeddings
#   - Apenas ingles -> menos preciso em portugues
modelo_alternativo = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
registro.info("Modelo all-MiniLM-L6-v2 carregado")

# Gerar indice alternativo com subamostragem (1000 trechos)
import numpy as np
trechos_amostra = [t for t in trechos if t["fonte"]=="fonte2"][:500]
if len(trechos_amostra) < 500:
    trechos_amostra = trechos[:500]

textos_amostra = [t["texto"] for t in trechos_amostra]
matriz_alt = modelo_alternativo.encode(textos_amostra, normalize_embeddings=True, batch_size=32)

import faiss
indice_alt = faiss.IndexFlatIP(matriz_alt.shape[1])
indice_alt.add(matriz_alt.astype(np.float32))

consulta = "atorvastatina interacao com ciclosporina"
embed_principal = modelo_principal.encode([consulta], normalize_embeddings=True)
embed_alt = modelo_alternativo.encode([consulta], normalize_embeddings=True)

# Score no espaco do modelo principal
d_principal, i_principal = indice_faiss.search(embed_principal.astype(np.float32), 3)
# Score no espaco do modelo alternativo
d_alt, i_alt = indice_alt.search(embed_alt.astype(np.float32), 3)

print("Consulta:", consulta)
print("\nModelo multilingue (paraphrase-multilingual-MiniLM-L12-v2):")
for rank, (dist, idx) in enumerate(zip(d_principal[0], i_principal[0]), 1):
    if 0 <= idx < len(trechos):
        print(f"  #{rank} [{trechos[idx]["medicamento"]}] score={dist:.4f}")

print("\nModelo ingles (all-MiniLM-L6-v2) -- subamostra:")
for rank, (dist, idx) in enumerate(zip(d_alt[0], i_alt[0]), 1):
    if 0 <= idx < len(trechos_amostra):
        print(f"  #{rank} [{trechos_amostra[idx]["medicamento"]}] score={dist:.4f}")

print("\nConclusao: O modelo multilingue e superior para bulas em portugues.")
registro.info("Comparacao de modelos: multilingue > ingles para PT-BR")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

2026-06-23 22:53:32 [INFO] Modelo all-MiniLM-L6-v2 carregado
Consulta: atorvastatina interacao com ciclosporina

Modelo multilingue (paraphrase-multilingual-MiniLM-L12-v2):
  #1 [pitavastatina calcica] score=0.8120
  #2 [olmesartana medoxomila besilato de anlodipino] score=0.8040
  #3 [sinergen] score=0.7955

Modelo ingles (all-MiniLM-L6-v2) -- subamostra:
  #1 [adriblastina rd] score=0.6360
  #2 [activelle] score=0.6259
  #3 [atropion] score=0.5993

Conclusao: O modelo multilingue e superior para bulas em portugues.
2026-06-23 22:53:33 [INFO] Comparacao de modelos: multilingue > ingles para PT-BR


## 3.6 Avaliacao: Recall@K e Analise de Falhas

**Recall@K:** Proporcao de consultas em que o medicamento relevante
aparece entre os K primeiros resultados.

**Falhas comuns:**
- Termos medicos nao cobertos pelo vocabulario de treino
- Sinonimia nao aprendida (ex: "anticoagulante" vs "varfarina")
- Trechos muito curtos ou muito longos
- Dominio especifico de bulas nao representado no fine-tuning


## 3.6.1 Dataset de Validacao (Ground Truth)


In [7]:
# Ground truth: consulta -> farmaco esperado no resultado
PARES_VALIDACAO = [
    {"consulta": "amoxicilina", "medicamento": "amoxicilina"},
    {"consulta": "dipirona", "medicamento": "dipirona"},
    {"consulta": "metformina", "medicamento": "metformina"},
    {"consulta": "sinvastatina", "medicamento": "sinvastatina"},
    {"consulta": "omeprazol", "medicamento": "omeprazol"},
    {"consulta": "losartana", "medicamento": "losartana"},
    {"consulta": "alopurinol", "medicamento": "alopurinol"},
    {"consulta": "atenolol", "medicamento": "atenolol"},
    {"consulta": "ibuprofeno", "medicamento": "ibuprofeno"},
    {"consulta": "warfaria", "medicamento": "warfarina"},   # nome similar
    {"consulta": "aspirina", "medicamento": "acido acetilsalicilico"},
    {"consulta": "paracetamol", "medicamento": "paracetamol"},
]

print("Dataset de validacao: %d pares" % len(PARES_VALIDACAO))
for p in PARES_VALIDACAO:
    print("  Consulta: %-30s  Esperado: %s" % (p["consulta"], p["medicamento"]))


Dataset de validacao: 12 pares
  Consulta: amoxicilina                     Esperado: amoxicilina
  Consulta: dipirona                        Esperado: dipirona
  Consulta: metformina                      Esperado: metformina
  Consulta: sinvastatina                    Esperado: sinvastatina
  Consulta: omeprazol                       Esperado: omeprazol
  Consulta: losartana                       Esperado: losartana
  Consulta: alopurinol                      Esperado: alopurinol
  Consulta: atenolol                        Esperado: atenolol
  Consulta: ibuprofeno                      Esperado: ibuprofeno
  Consulta: warfaria                        Esperado: warfarina
  Consulta: aspirina                        Esperado: acido acetilsalicilico
  Consulta: paracetamol                     Esperado: paracetamol


In [8]:
def calcular_recall_at_k(pares_validacao, modelo_embeddings, indice_faiss, trechos, k=5):
    acertos = 0
    total = len(pares_validacao)
    for par in pares_validacao:
        medicamento_esperado = par["medicamento"]
        embed = modelo_embeddings.encode([par["consulta"]], normalize_embeddings=True).astype(np.float32)
        _, indices = indice_faiss.search(embed, k)
        meds_recuperados = [trechos[i]["medicamento"] for i in indices[0] if 0 <= i < len(trechos)]
        if medicamento_esperado in meds_recuperados:
            acertos += 1
    return acertos / total, acertos, total



In [9]:
recall, acertos, total = calcular_recall_at_k(
    PARES_VALIDACAO, modelo_principal, indice_faiss, trechos, k=5
)

print("AVALIACAO DE RECALL@5".center(60, "="))
print("  Recall@5: {0:.1%}  ({1}/{2})".format(recall, acertos, total))
print("  Acertos por par:")
for par in PARES_VALIDACAO:
    embed = modelo_principal.encode([par["consulta"]], normalize_embeddings=True).astype(np.float32)
    _, indices = indice_faiss.search(embed, 5)
    meds = [trechos[i]["medicamento"] for i in indices[0] if 0 <= i < len(trechos)]
    hit = par["medicamento"] in meds
    status = "OK" if hit else "FALHA"
    print("    [{0}] {1}... -> {2}".format(status, par["consulta"][:50], meds[:3]))

print("\nANALISE DE FALHAS:")
falhas = 0
for par in PARES_VALIDACAO:
    embed = modelo_principal.encode([par["consulta"]], normalize_embeddings=True).astype(np.float32)
    _, indices = indice_faiss.search(embed, 5)
    meds = [trechos[i]["medicamento"] for i in indices[0] if 0 <= i < len(trechos)]
    if par["medicamento"] not in meds:
        falhas += 1
        if falhas <= 3:
            print("    Consulta: {0}... / Esperado: {1}".format(par["consulta"][:50], par["medicamento"]))
            print("    Motivo provavel: vocabulario fora do dominio ou sinonimia nao aprendida")
if falhas == 0:
    print("  Nenhuma falha em recall@5")
else:
    print("  {0} falhas identificadas".format(falhas))

registro.info("Recall@5: %.2f (%d/%d)", recall, acertos, total)



===================AVALIACAO DE RECALL@5====================
  Recall@5: 25.0%  (3/12)
  Acertos por par:
    [FALHA] amoxicilina... -> ['omeprazol', 'omeprazol', 'omeprazol']
    [FALHA] dipirona... -> ['dipirona sodica', 'dipirona monoidratada', 'dipirona monoidratada']
    [FALHA] metformina... -> ['topiramato', 'indatrat sr', 'indapamida ems']
    [OK] sinvastatina... -> ['sinvastatina', 'levetiracetam', 'oxalato de escitalopram']
    [OK] omeprazol... -> ['omeprazol', 'omeprazol', 'omeprazol']
    [FALHA] losartana... -> ['besilato de anlodipino atenolol', 'metronidazol', 'dicloridrato de pramipexol']
    [FALHA] alopurinol... -> ['lotensin h', 'diovan triplo', 'perindopril erbumina indapamida']
    [FALHA] atenolol... -> ['besilato de anlodipino atenolol', 'nitazoxanida', 'omeprazol']
    [FALHA] ibuprofeno... -> ['ibuprofan', 'ibuprofeno comprimido cimed', 'ibuprofeno legrand']
    [FALHA] warfaria... -> ['warfarin', 'warfarin', 'citalopram']
    [FALHA] aspirina... -> ['tadalaf

## 3.7 Conclusao e Decisoes Tecnicas

### Decisoes

- **Modelo:** paraphrase-multilingual-MiniLM-L12-v2 (384d, multilingue)
- **Indice:** FAISS IndexFlatIP com normalizacao L2
- **Estrategia:** Busca hibrida 60% semantica + 40% BM25
- **Top-K:** 5 resultados por consulta

### Justificativa

O modelo multilingue generaliza melhor para PT-BR do que modelos
ingleses. FAISS IndexFlatIP e suficiente para <100k vetores.
A busca hibrida mitiga limitacoes de cada abordagem isolada.

### Limites

- Chunking em sentencas pode perder contexto entre sentencas
- FAISS nao suporta metadata diretamente (resolvido com mapeamento)
- Sem reranking (BM25 + semantic ja funciona razoavelmente)


In [10]:
registro.info("=" * 60)
registro.info("Caderno 03 concluido.")
registro.info("  Trechos indexados: %d", len(trechos))
registro.info("  Recall@5: %.2f", recall)
registro.info("Fim: %s", datetime.now().isoformat())
print("=" * 60)
print("Caderno 03 -- Embeddings e Busca Vetorial: CONCLUIDO")
print(f"Modelo: paraphrase-multilingual-MiniLM-L12-v2 (384d)")
print(f"Indice: FAISS IndexFlatIP + BM25 hibrido")
print(f"Recall@5: {recall:.1%}")


2026-06-23 22:53:33 [INFO] ============================================================
2026-06-23 22:53:33 [INFO] Caderno 03 concluido.
2026-06-23 22:53:33 [INFO]   Trechos indexados: 116152
2026-06-23 22:53:33 [INFO]   Recall@5: 0.25
2026-06-23 22:53:33 [INFO] Fim: 2026-06-23T22:53:33.838131
Caderno 03 -- Embeddings e Busca Vetorial: CONCLUIDO
Modelo: paraphrase-multilingual-MiniLM-L12-v2 (384d)
Indice: FAISS IndexFlatIP + BM25 hibrido
Recall@5: 25.0%
